# Day 2 — Silver and Quality Engineering

This notebook preserves the original Silver-layer flow and adds the missing Day 2 controls:
- DQ rule catalogue with rule ID, severity, and expected action
- Raw-value preservation before parsing/casting
- Deterministic encounter deduplication and duplicate audit counts
- CDC conflict checks and deterministic current-state selection
- Identifier masking in sample outputs
- Unit tests for valid, invalid, duplicate, late, and out-of-order cases

**Important source-data limitation:** the listed encounter and CDC columns do not establish explicit admission, transfer, and discharge event types with valid start/end timestamps. The notebook therefore does not invent stay intervals. The relevant section documents what source fields are needed before that business logic can be safely implemented.

**Before running:** verify the Bronze schema and reference table names in Catalog Explorer. Rejected CDC and secondary records are displayed as in-memory DataFrames only; no separate rejected tables are created. No metric thresholds are configured.

## 1. Imports and table configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

CATALOG = "capstone_8hospital"

# CHANGE only if Catalog Explorer shows different schemas/table names.
BRONZE_SCHEMA = "hospital_schema"
SILVER_SCHEMA = "silver_schema"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

BRONZE_ENCOUNTER_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_encounter_events"
BRONZE_CDC_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_status_cdc"
BRONZE_SECONDARY_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_secondary_events"

# Confirm these two names and schemas against your actual reference tables.
BRONZE_DEPARTMENT_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.department_bronze"
BRONZE_PATIENT_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.patient_bronze"

SILVER_ENCOUNTER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_encounter_events"
SILVER_REJECTED_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_rejected_records"
SILVER_DQ_RULE_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_dq_rule_catalogue"
SILVER_STATUS_HISTORY_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_status_history"
SILVER_CURRENT_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_encounter_current"
SILVER_SECONDARY_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_secondary_events"



## 2. Create the Data Quality Rules Catalogue table

The catalogue makes the rules visible in Databricks. No bill-amount or wait-time threshold is configured in this notebook.

In [0]:
rule_rows = [
    ("DQ-01", "encounter_id", "Required identifier is present", "ERROR", "REJECT", "Missing or blank encounter_id"),
    ("DQ-02", "event_timestamp", "Timestamp is valid", "ERROR", "REJECT", "Missing or unparseable event_timestamp"),
    ("DQ-03", "bill_amount", "Bill amount is numeric and greater than zero", "ERROR", "REJECT", "Invalid bill_amount"),
    ("DQ-04", "wait_minutes", "Wait minutes is numeric and non-negative", "ERROR", "REJECT", "Invalid wait_minutes"),
    ("DQ-05", "department_id", "Department key exists in reference table", "ERROR", "REJECT", "Unmatched department_id"),
    ("DQ-06", "patient_id", "Patient key exists in reference table", "ERROR", "REJECT", "Unmatched patient_id"),
    ("DQ-07", "encounter_id", "Duplicate encounter key after quality checks", "WARNING", "QUARANTINE_EXTRA", "Duplicate encounter_id; one row retained"),
    ("DQ-08", "optional categorical fields", "Missing optional category is retained and flagged", "WARNING", "FLAG", "Missing optional categorical value"),
]

rule_schema = T.StructType([
    T.StructField("rule_id", T.StringType(), False),
    T.StructField("field_name", T.StringType(), False),
    T.StructField("rule_description", T.StringType(), False),
    T.StructField("severity", T.StringType(), False),
    T.StructField("action", T.StringType(), False),
    T.StructField("failure_reason", T.StringType(), False),
])

RULE_IDS = {row[0]: row[0] for row in rule_rows}
dq_rules = spark.createDataFrame(rule_rows, schema=rule_schema).withColumn(
    "updated_at", F.current_timestamp()
)

display(dq_rules.orderBy("rule_id"))

(dq_rules.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(SILVER_DQ_RULE_TABLE))

print("Saved rule catalogue:", SILVER_DQ_RULE_TABLE)

rule_id,field_name,rule_description,severity,action,failure_reason,updated_at
DQ-01,encounter_id,Required identifier is present,ERROR,REJECT,Missing or blank encounter_id,2026-11-03T19:14:11.853Z
DQ-02,event_timestamp,Timestamp is valid,ERROR,REJECT,Missing or unparseable event_timestamp,2026-11-03T19:14:11.853Z
DQ-03,bill_amount,Bill amount is numeric and greater than zero,ERROR,REJECT,Invalid bill_amount,2026-11-03T19:14:11.853Z
DQ-04,wait_minutes,Wait minutes is numeric and non-negative,ERROR,REJECT,Invalid wait_minutes,2026-11-03T19:14:11.853Z
DQ-05,department_id,Department key exists in reference table,ERROR,REJECT,Unmatched department_id,2026-11-03T19:14:11.853Z
DQ-06,patient_id,Patient key exists in reference table,ERROR,REJECT,Unmatched patient_id,2026-11-03T19:14:11.853Z
DQ-07,encounter_id,Duplicate encounter key after quality checks,WARNING,QUARANTINE_EXTRA,Duplicate encounter_id; one row retained,2026-11-03T19:14:11.853Z
DQ-08,optional categorical fields,Missing optional category is retained and flagged,WARNING,FLAG,Missing optional categorical value,2026-11-03T19:14:11.853Z


Saved rule catalogue: capstone_8hospital.silver_schema.silver_dq_rule_catalogue


## 3. Read Bronze tables and inspect schemas

In [0]:
encounters = spark.table(BRONZE_ENCOUNTER_TABLE)
cdc = spark.table(BRONZE_CDC_TABLE)
secondary = spark.table(BRONZE_SECONDARY_TABLE)
departments = spark.table(BRONZE_DEPARTMENT_TABLE)
patients = spark.table(BRONZE_PATIENT_TABLE)

encounters.printSchema()
cdc.printSchema()
secondary.printSchema()
departments.printSchema()
patients.printSchema()

display(spark.createDataFrame(
    [
        ("Bronze encounter events", encounters.count()),
        ("Bronze status CDC", cdc.count()),
        ("Bronze secondary events", secondary.count()),
        ("Department reference", departments.count()),
        ("Patient reference", patients.count()),
    ],
    ["table_name", "row_count"]
))

root
 |-- encounter_id: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- ingest_timestamp: timestamp (nullable = true)
 |-- department_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- visit_type: string (nullable = true)
 |-- encounter_status: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- bill_amount: double (nullable = true)
 |-- wait_minutes: integer (nullable = true)
 |-- doctor_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_schema_version: string (nullable = true)
 |-- _rescued_data: string (nullable = true)

root
 |-- encounter_id: string (nullable = true)
 |-- change_timestamp: timestamp (nullable = true)
 |-- previous_status: string (nullable = true)
 |-- new_status: string (nullable = true)
 |-- change_reason: string (nullable = true)
 |-- source_sequence: integer (nullable = true)
 |-- source_system: string (nullable = true)

root
 |-- event_id: string (null

table_name,row_count
Bronze encounter events,11540
Bronze status CDC,520
Bronze secondary events,310
Department reference,110
Patient reference,150


## 4. Cast and standardize encounter data

In [0]:
# Preserve source values before any casting so failed parses remain traceable.
source_columns = encounters.columns
typed = encounters

# Add explicit raw-value columns (only when the source column exists).
for c in ["event_timestamp", "ingest_timestamp", "bill_amount", "wait_minutes"]:
    if c in encounters.columns:
        typed = typed.withColumn(f"{c}_raw", F.col(c).cast("string"))

# Stable tie-breaker based on the complete original source row.
typed = typed.withColumn(
    "_source_row_hash",
    F.sha2(F.to_json(F.struct(*[F.col(c) for c in source_columns])), 256)
)

typed = (
    typed
    .withColumn("encounter_id", F.trim(F.col("encounter_id").cast("string")))
    .withColumn("patient_id", F.trim(F.col("patient_id").cast("string")))
    .withColumn("department_id", F.trim(F.col("department_id").cast("string")))
    .withColumn("event_timestamp", F.to_timestamp("event_timestamp"))
    .withColumn("ingest_timestamp", F.to_timestamp("ingest_timestamp"))
    .withColumn("bill_amount", F.col("bill_amount").cast("double"))
    .withColumn("wait_minutes", F.col("wait_minutes").cast("int"))
)

print("Typed rows:", typed.count())
display(typed.select(
    "encounter_id", "event_timestamp_raw", "event_timestamp",
    "bill_amount_raw", "bill_amount", "wait_minutes_raw", "wait_minutes"
).limit(10))

Typed rows: 11540


encounter_id,event_timestamp_raw,event_timestamp,bill_amount_raw,bill_amount,wait_minutes_raw,wait_minutes
ENCOUNTER-0011352,2026-08-01 12:46:00,2026-08-01T12:46:00.000Z,1299.92,1299.92,60,60
ENCOUNTER-0011375,2026-08-01 14:17:00,2026-08-01T14:17:00.000Z,1162.34,1162.34,14,14
ENCOUNTER-0007053,2026-07-20 14:08:00,2026-07-20T14:08:00.000Z,72.97,72.97,123,123
ENCOUNTER-0008647,2026-07-25 00:25:00,2026-07-25T00:25:00.000Z,961.5,961.5,151,151
ENCOUNTER-0005083,2026-07-15 02:48:00,2026-07-15T02:48:00.000Z,513.87,513.87,149,149
ENCOUNTER-0000073,2026-07-01 04:49:00,2026-07-01T04:49:00.000Z,1262.76,1262.76,44,44
ENCOUNTER-0003374,2026-07-10 08:53:00,2026-07-10T08:53:00.000Z,586.49,586.49,51,51
ENCOUNTER-0005078,2026-07-15 02:30:00,2026-07-15T02:30:00.000Z,177.55,177.55,32,32
ENCOUNTER-0004618,2026-07-13 19:50:00,2026-07-13T19:50:00.000Z,1373.1,1373.1,2,2
ENCOUNTER-0003298,2026-07-10 03:50:00,2026-07-10T03:50:00.000Z,1153.18,1153.18,146,146


## 5. Apply DQ-01 to DQ-06 using the rule IDs from the catalogue

In [0]:
# Rule IDs are defined in silver_dq_rule_catalogue above and reused in the checks.
# Preserve the first applicable rejection reason for each row.

checked = (
    typed
    .withColumn(
        "rule_id",
        F.when(F.col("encounter_id").isNull() | (F.col("encounter_id") == ""), F.lit(RULE_IDS["DQ-01"]))
         .when(F.col("event_timestamp").isNull(), F.lit(RULE_IDS["DQ-02"]))
         .when(F.col("bill_amount").isNull() | (F.col("bill_amount") <= 0), F.lit(RULE_IDS["DQ-03"]))
         .when(F.col("wait_minutes").isNull() | (F.col("wait_minutes") < 0), F.lit(RULE_IDS["DQ-04"]))
    )
    .withColumn(
        "rejection_reason",
        F.when(F.col("rule_id") == "DQ-01", F.lit("Missing or blank encounter_id"))
         .when(F.col("rule_id") == "DQ-02", F.lit("Missing or unparseable event_timestamp; raw value preserved in event_timestamp_raw"))
         .when(F.col("rule_id") == "DQ-03", F.lit("bill_amount must be numeric and greater than zero; inspect bill_amount_raw"))
         .when(F.col("rule_id") == "DQ-04", F.lit("wait_minutes must be numeric and non-negative; inspect wait_minutes_raw"))
    )
)

department_keys = departments.select(F.trim(F.col("department_id").cast("string")).alias("department_id")).dropDuplicates()
patient_keys = patients.select(F.trim(F.col("patient_id").cast("string")).alias("patient_id")).dropDuplicates()

with_refs = (
    checked
    .join(department_keys.withColumn("_department_found", F.lit(True)), "department_id", "left")
    .join(patient_keys.withColumn("_patient_found", F.lit(True)), "patient_id", "left")
    .withColumn(
        "rule_id",
        F.when(F.col("rule_id").isNotNull(), F.col("rule_id"))
         .when(F.col("_department_found").isNull(), F.lit(RULE_IDS["DQ-05"]))
         .when(F.col("_patient_found").isNull(), F.lit(RULE_IDS["DQ-06"]))
    )
    .withColumn(
        "rejection_reason",
        F.when(F.col("rejection_reason").isNotNull(), F.col("rejection_reason"))
         .when(F.col("rule_id") == "DQ-05", F.lit("department_id not found in department reference"))
         .when(F.col("rule_id") == "DQ-06", F.lit("patient_id not found in patient reference"))
    )
    .withColumn("rejected_at", F.current_timestamp())
)

rejected_basic = with_refs.filter(F.col("rule_id").isNotNull())
valid_basic = with_refs.filter(F.col("rule_id").isNull())

print("Rows before deduplication:", valid_basic.count())
print("Rejected by DQ-01 to DQ-06:", rejected_basic.count())

Rows before deduplication: 11540
Rejected by DQ-01 to DQ-06: 0


## 6. Apply DQ-07 deduplication and DQ-08 missing-category flag

This follows the guide's business-key assumption: keep one row per `encounter_id`, choosing the most recently ingested row. Confirm that the official task defines duplicates by encounter key rather than exact-row equality.

In [0]:
# Deterministic tie-breaker:
# 1) latest ingest_timestamp, 2) stable source-file path when available,
# 3) stable hash of the complete original source row.
order_columns = [F.col("ingest_timestamp").desc_nulls_last()]
if "_source_file" in valid_basic.columns:
    order_columns.append(F.col("_source_file").asc_nulls_last())
order_columns.append(F.col("_source_row_hash").asc_nulls_last())

dedup_window = Window.partitionBy("encounter_id").orderBy(*order_columns)
ranked = valid_basic.withColumn("_row_num", F.row_number().over(dedup_window))

duplicate_rows = (
    ranked.filter(F.col("_row_num") > 1)
    .withColumn("rule_id", F.lit(RULE_IDS["DQ-07"]))
    .withColumn("rejection_reason", F.lit("Duplicate encounter_id; latest ingestion then stable source tie-breaker retained"))
    .withColumn("rejected_at", F.current_timestamp())
)

deduplicated = ranked.filter(F.col("_row_num") == 1).drop(
    "_row_num", "_department_found", "_patient_found"
)

rejected_final = rejected_basic.unionByName(duplicate_rows, allowMissingColumns=True)

# DQ-08: retain records with missing optional categories and flag them.
optional_candidates = [c for c in ["visit_type", "encounter_status", "priority", "source_system"] if c in deduplicated.columns]
missing_optional_expr = None
for c in optional_candidates:
    expr = F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == "")
    missing_optional_expr = expr if missing_optional_expr is None else (missing_optional_expr | expr)

if missing_optional_expr is None:
    missing_optional_expr = F.lit(False)

silver_encounters = (
    deduplicated
    .withColumn("missing_optional_category_flag", missing_optional_expr)
    .withColumn(
        "dq_status",
        F.when(F.col("missing_optional_category_flag"), F.lit("FLAGGED"))
         .otherwise(F.lit("VALID"))
    )
)

input_count = encounters.count()
valid_before_dedup_count = valid_basic.count()
silver_count = silver_encounters.count()
duplicate_count = duplicate_rows.count()
rejected_mandatory_count = rejected_basic.count()

print("Bronze encounter rows:", input_count)
print("Rejected for DQ-01 to DQ-06:", rejected_mandatory_count)
print("Valid rows before deduplication:", valid_before_dedup_count)
print("Duplicate rows quarantined:", duplicate_count)
print("Silver encounter rows after deduplication:", silver_count)
print("Audit reconciliation (Bronze = mandatory rejects + valid-before-dedup):",
      input_count == rejected_mandatory_count + valid_before_dedup_count)

display(silver_encounters.groupBy("dq_status", "missing_optional_category_flag").count())
display(duplicate_rows.select("encounter_id", "rule_id", "rejection_reason", "ingest_timestamp").limit(50))

Bronze encounter rows: 11540
Rejected for DQ-01 to DQ-06: 0
Valid rows before deduplication: 11540
Duplicate rows quarantined: 40
Silver encounter rows after deduplication: 11500
Audit reconciliation (Bronze = mandatory rejects + valid-before-dedup): True


dq_status,missing_optional_category_flag,count
VALID,false,11488
FLAGGED,true,12


encounter_id,rule_id,rejection_reason,ingest_timestamp
ENCOUNTER-0000501,DQ-07,Duplicate encounter_id; latest ingestion then stable source tie-breaker retained,2026-07-02T09:36:00.000Z
ENCOUNTER-0000502,DQ-07,Duplicate encounter_id; latest ingestion then stable source tie-breaker retained,2026-07-02T09:47:00.000Z
ENCOUNTER-0000503,DQ-07,Duplicate encounter_id; latest ingestion then stable source tie-breaker retained,2026-07-02T09:48:00.000Z
ENCOUNTER-0000504,DQ-07,Duplicate encounter_id; latest ingestion then stable source tie-breaker retained,2026-07-02T09:53:00.000Z
ENCOUNTER-0000505,DQ-07,Duplicate encounter_id; latest ingestion then stable source tie-breaker retained,2026-07-02T09:56:00.000Z
ENCOUNTER-0000506,DQ-07,Duplicate encounter_id; latest ingestion then stable source tie-breaker retained,2026-07-02T10:04:00.000Z
ENCOUNTER-0000507,DQ-07,Duplicate encounter_id; latest ingestion then stable source tie-breaker retained,2026-07-02T09:49:00.000Z
ENCOUNTER-0000508,DQ-07,Duplicate encounter_id; latest ingestion then stable source tie-breaker retained,2026-07-02T10:05:00.000Z
ENCOUNTER-0000509,DQ-07,Duplicate encounter_id; latest ingestion then stable source tie-breaker retained,2026-07-02T10:10:00.000Z
ENCOUNTER-0000510,DQ-07,Duplicate encounter_id; latest ingestion then stable source tie-breaker retained,2026-07-02T10:05:00.000Z


## 7. Write Silver encounter and rejected-record tables

In [0]:
(silver_encounters.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(SILVER_ENCOUNTER_TABLE))

(rejected_final.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(SILVER_REJECTED_TABLE))

print("Saved:", SILVER_ENCOUNTER_TABLE)
print("Saved:", SILVER_REJECTED_TABLE)

Saved: capstone_8hospital.silver_schema.silver_encounter_events
Saved: capstone_8hospital.silver_schema.silver_rejected_records


## 8. CDC: inspect real columns before processing

Your guide's CDC code assumes `event_timestamp`, but the source described in the project uses `change_timestamp`. This cell checks the actual schema and uses `change_timestamp` when present. It does not treat operational workflow statuses as admission/transfer/discharge events.

In [0]:
cdc.printSchema()
print("CDC columns:", cdc.columns)
display(cdc.limit(10))

# Use the source's effective/change timestamp when present.
if "effective_timestamp" in cdc.columns:
    cdc_time_col = "effective_timestamp"
elif "change_timestamp" in cdc.columns:
    cdc_time_col = "change_timestamp"
elif "event_timestamp" in cdc.columns:
    cdc_time_col = "event_timestamp"
else:
    raise ValueError("CDC table has no effective_timestamp, change_timestamp, or event_timestamp. Check the source schema.")

required_cdc_cols = ["encounter_id", "new_status", "source_sequence", cdc_time_col]
missing_cdc_cols = [c for c in required_cdc_cols if c not in cdc.columns]
if missing_cdc_cols:
    raise ValueError(f"CDC source is missing required columns: {missing_cdc_cols}")

cdc_typed = (
    cdc
    .withColumn("change_timestamp_raw", F.col(cdc_time_col).cast("string"))
    .withColumn("encounter_id", F.trim(F.col("encounter_id").cast("string")))
    .withColumn("source_sequence", F.col("source_sequence").cast("long"))
    .withColumn("status_changed_at", F.to_timestamp(F.col(cdc_time_col)))
    .withColumn(
        "_cdc_event_hash",
        F.sha2(F.to_json(F.struct(*[F.col(c) for c in cdc.columns])), 256)
    )
)

cdc_checked = (
    cdc_typed
    .withColumn(
        "rule_id",
        F.when(F.col("encounter_id").isNull() | (F.col("encounter_id") == ""), F.lit("CDC-01"))
         .when(F.col("new_status").isNull() | (F.trim(F.col("new_status").cast("string")) == ""), F.lit("CDC-02"))
         .when(F.col("source_sequence").isNull(), F.lit("CDC-03"))
         .when(F.col("status_changed_at").isNull(), F.lit("CDC-04"))
    )
    .withColumn(
        "rejection_reason",
        F.when(F.col("rule_id") == "CDC-01", F.lit("Missing encounter_id"))
         .when(F.col("rule_id") == "CDC-02", F.lit("Missing new_status"))
         .when(F.col("rule_id") == "CDC-03", F.lit("Missing or invalid source_sequence"))
         .when(F.col("rule_id") == "CDC-04", F.lit("Missing or invalid effective/change timestamp"))
    )
    .withColumn("rejected_at", F.current_timestamp())
)

cdc_rejected = cdc_checked.filter(F.col("rule_id").isNotNull())
cdc_valid = cdc_checked.filter(F.col("rule_id").isNull())

print("Valid CDC rows:", cdc_valid.count())
print("Rejected CDC rows (in-memory only; no rejected table):", cdc_rejected.count())
display(cdc_rejected.select("encounter_id", "source_sequence", "new_status", "change_timestamp_raw", "rule_id", "rejection_reason").limit(50))

root
 |-- encounter_id: string (nullable = true)
 |-- change_timestamp: timestamp (nullable = true)
 |-- previous_status: string (nullable = true)
 |-- new_status: string (nullable = true)
 |-- change_reason: string (nullable = true)
 |-- source_sequence: integer (nullable = true)
 |-- source_system: string (nullable = true)

CDC columns: ['encounter_id', 'change_timestamp', 'previous_status', 'new_status', 'change_reason', 'source_sequence', 'source_system']


encounter_id,change_timestamp,previous_status,new_status,change_reason,source_sequence,source_system
ENCOUNTER-0000001,2026-08-10T00:00:00.000Z,PENDING,CANCELLED,OPERATIONS_UPDATE,1,ADLS_CDC
ENCOUNTER-0000018,2026-08-10T00:01:00.000Z,OPEN,COMPLETED,CUSTOMER_ACTION,2,ADLS_CDC
ENCOUNTER-0000035,2026-08-10T00:02:00.000Z,COMPLETED,REVERSED,SYSTEM_REVIEW,3,ADLS_CDC
ENCOUNTER-0000052,2026-08-10T00:03:00.000Z,PENDING,COMPLETED,OPERATIONS_UPDATE,4,ADLS_CDC
ENCOUNTER-0000069,2026-08-10T00:04:00.000Z,COMPLETED,COMPLETED,SYSTEM_REVIEW,5,ADLS_CDC
ENCOUNTER-0000086,2026-08-10T00:05:00.000Z,FAILED,COMPLETED,CUSTOMER_ACTION,6,ADLS_CDC
ENCOUNTER-0000103,2026-08-10T00:06:00.000Z,COMPLETED,REVERSED,SYSTEM_REVIEW,7,ADLS_CDC
ENCOUNTER-0000120,2026-08-10T00:07:00.000Z,FAILED,CANCELLED,SYSTEM_REVIEW,8,ADLS_CDC
ENCOUNTER-0000137,2026-08-10T00:08:00.000Z,OPEN,CANCELLED,SYSTEM_REVIEW,9,ADLS_CDC
ENCOUNTER-0000154,2026-08-10T00:09:00.000Z,COMPLETED,CANCELLED,OPERATIONS_UPDATE,10,ADLS_CDC


Valid CDC rows: 520
Rejected CDC rows (in-memory only; no rejected table): 0


encounter_id,source_sequence,new_status,change_timestamp_raw,rule_id,rejection_reason


## 9. Build status history and latest status

The sequence is used first to select the latest state. Duplicate `(encounter_id, source_sequence)` pairs are displayed for inspection before one row is retained.

In [0]:
# Exact replayed rows are safe to collapse. Do not arbitrarily discard conflicting
# events that share (encounter_id, source_sequence).
cdc_exact_dedup = cdc_valid.dropDuplicates(cdc_valid.columns)

sequence_conflicts = (
    cdc_exact_dedup
    .groupBy("encounter_id", "source_sequence")
    .agg(
        F.countDistinct(F.struct("new_status", "status_changed_at")).alias("distinct_event_versions")
    )
    .filter(F.col("distinct_event_versions") > 1)
)

print("Conflicting encounter/sequence keys:", sequence_conflicts.count())
display(sequence_conflicts.orderBy("encounter_id", "source_sequence"))

# Keep the full valid event history (after exact replay deduplication).
cdc_history = cdc_exact_dedup

(cdc_history.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(SILVER_STATUS_HISTORY_TABLE))

# Deterministic ordering: highest source_sequence, then latest effective/change time,
# then stable hash. Conflicts are surfaced above rather than silently removed.
status_window = Window.partitionBy("encounter_id").orderBy(
    F.col("source_sequence").desc(),
    F.col("status_changed_at").desc(),
    F.col("_cdc_event_hash").desc()
)

latest_status = (
    cdc_history
    .withColumn("_rn", F.row_number().over(status_window))
    .filter(F.col("_rn") == 1)
    .select(
        "encounter_id",
        F.col("new_status").alias("current_status"),
        F.col("source_sequence").alias("current_source_sequence"),
        F.col("status_changed_at")
    )
)

base_encounters = spark.table(SILVER_ENCOUNTER_TABLE)
for col_name in ["current_status", "current_source_sequence", "status_changed_at"]:
    if col_name in base_encounters.columns:
        base_encounters = base_encounters.drop(col_name)

current_encounters = base_encounters.join(latest_status, "encounter_id", "left")

(current_encounters.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(SILVER_CURRENT_TABLE))

print("Status history rows:", spark.table(SILVER_STATUS_HISTORY_TABLE).count())
print("Current encounter rows:", spark.table(SILVER_CURRENT_TABLE).count())

# Mask identifiers in sample outputs; full identifiers remain in governed tables for joins.
display(
    spark.table(SILVER_CURRENT_TABLE)
    .select(
        F.concat(F.lit("***"), F.substring("encounter_id", -4, 4)).alias("encounter_id_masked"),
        "current_status", "current_source_sequence", "status_changed_at"
    )
    .orderBy("current_source_sequence", ascending=False)
    .limit(30)
)

Conflicting encounter/sequence keys: 0


encounter_id,source_sequence,distinct_event_versions


Status history rows: 520
Current encounter rows: 11500


encounter_id_masked,current_status,current_source_sequence,status_changed_at
***8824,COMPLETED,520,2026-08-10T08:39:00.000Z
***8807,REVERSED,519,2026-08-10T08:38:00.000Z
***8790,REVERSED,518,2026-08-10T08:37:00.000Z
***8773,CANCELLED,517,2026-08-10T08:36:00.000Z
***8756,REVERSED,516,2026-08-10T08:35:00.000Z
***8739,CANCELLED,515,2026-08-10T08:34:00.000Z
***8722,CANCELLED,514,2026-08-10T08:33:00.000Z
***8705,COMPLETED,513,2026-08-10T08:32:00.000Z
***8688,CANCELLED,512,2026-08-10T08:31:00.000Z
***8671,REVERSED,511,2026-08-10T08:30:00.000Z


## 10. Secondary operational events

The exact secondary-event schema and rule requirements must be checked first. This conservative starter validates that the event has an encounter key and that the key exists in Silver encounters. Add other required checks from the official task before treating this table as complete.

In [0]:
secondary.printSchema()
display(secondary.limit(10))

silver_encounter_keys = spark.table(SILVER_ENCOUNTER_TABLE).select("encounter_id").dropDuplicates()

if "encounter_id" not in secondary.columns:
    raise ValueError("Secondary-event table does not contain encounter_id. Update the key after checking printSchema().")

secondary_checked = (
    secondary
    .withColumn("encounter_id", F.trim(F.col("encounter_id").cast("string")))
    .join(silver_encounter_keys.withColumn("_encounter_found", F.lit(True)), "encounter_id", "left")
    .withColumn(
        "rule_id",
        F.when(F.col("encounter_id").isNull() | (F.col("encounter_id") == ""), F.lit("SEC-01"))
         .when(F.col("_encounter_found").isNull(), F.lit("SEC-02"))
    )
    .withColumn(
        "rejection_reason",
        F.when(F.col("rule_id") == "SEC-01", F.lit("Missing encounter_id"))
         .when(F.col("rule_id") == "SEC-02", F.lit("Encounter key not found in Silver encounters"))
    )
    .withColumn("rejected_at", F.current_timestamp())
)

secondary_rejected = secondary_checked.filter(F.col("rule_id").isNotNull())
silver_secondary = secondary_checked.filter(F.col("rule_id").isNull()).drop("_encounter_found")

(silver_secondary.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_SECONDARY_TABLE))

print("Secondary Bronze rows:", secondary.count())
print("Secondary Silver rows:", spark.table(SILVER_SECONDARY_TABLE).count())
print("Secondary rejected rows (in-memory only; no rejected table):", secondary_rejected.count())

display(
    secondary_rejected.select(
        F.concat(F.lit("***"), F.substring("encounter_id", -4, 4)).alias("encounter_id_masked"),
        "rule_id", "rejection_reason"
    ).limit(50)
)

root
 |-- event_id: string (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- related_entity_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_value: double (nullable = true)
 |-- source_system: string (nullable = true)



event_id,encounter_id,event_timestamp,related_entity_id,event_type,event_value,source_system
SEC-000001,ENCOUNTER-0000001,2026-07-01T00:01:00.000Z,DEPARTMENT_ID-0001,ALERT,65.11,ADLS_AUX
SEC-000002,ENCOUNTER-0000032,2026-07-01T02:06:00.000Z,DEPARTMENT_ID-0032,ALERT,48.39,ADLS_AUX
SEC-000003,ENCOUNTER-0000063,2026-07-01T04:08:00.000Z,DEPARTMENT_ID-0063,CHECK,50.19,ADLS_AUX
SEC-000004,ENCOUNTER-0000094,2026-07-01T06:14:00.000Z,DEPARTMENT_ID-0094,CHECK,96.95,ADLS_AUX
SEC-000005,ENCOUNTER-0000125,2026-07-01T08:16:00.000Z,DEPARTMENT_ID-0015,ALERT,49.92,ADLS_AUX
SEC-000006,ENCOUNTER-0000156,2026-07-01T10:20:00.000Z,DEPARTMENT_ID-0046,CHECK,69.62,ADLS_AUX
SEC-000007,ENCOUNTER-0000187,2026-07-01T12:24:00.000Z,DEPARTMENT_ID-0077,REVIEW,95.45,ADLS_AUX
SEC-000008,ENCOUNTER-0000218,2026-07-01T14:29:00.000Z,DEPARTMENT_ID-0108,REVIEW,92.51,ADLS_AUX
SEC-000009,ENCOUNTER-0000249,2026-07-01T16:33:00.000Z,DEPARTMENT_ID-0029,ALERT,23.69,ADLS_AUX
SEC-000010,ENCOUNTER-0000280,2026-07-01T18:37:00.000Z,DEPARTMENT_ID-0060,ALERT,23.53,ADLS_AUX


Secondary Bronze rows: 310
Secondary Silver rows: 310
Secondary rejected rows (in-memory only; no rejected table): 0


encounter_id_masked,rule_id,rejection_reason


## 11. Admission, transfer, discharge, stay intervals, and identifier protection

The encounter columns currently documented for this project include `event_timestamp`, `encounter_status`, and operational fields. The CDC status examples include operational workflow values such as `COMPLETED`, `CANCELLED`, and `REVERSED`. These fields alone do **not** prove which events are admission, transfer, or discharge.

Therefore, this notebook intentionally does not calculate hospital stay intervals from those statuses. To implement this requirement, confirm that the source provides:
- a reliable event type (admission / transfer / discharge),
- an encounter or visit key,
- an effective timestamp for each event,
- and the expected handling of missing or out-of-order events.

Once those fields and rules are confirmed, construct intervals by pairing each admission with its valid discharge and splitting intervals at transfers according to the project's definition. Reject or flag end-before-start intervals rather than silently correcting them.

Identifier protection: sample outputs in the CDC and secondary sections mask encounter IDs. Keep direct identifiers limited to authorized tables and users; apply the project's approved access controls or masking policy to patient identifiers before wider sharing.

## 12. Unit tests for valid, invalid, duplicate, late, and out-of-order cases

These tests use small synthetic data and do not write test rows into project tables.

In [0]:
# Unit tests use synthetic rows only.
from datetime import datetime

# Test A: valid and invalid mandatory fields.
test_quality = spark.createDataFrame([
    ("E-VALID", "2026-01-01 10:00:00", "100.0", "15"),
    ("", "2026-01-01 10:05:00", "100.0", "15"),
    ("E-BAD-AMOUNT", "2026-01-01 10:10:00", "not-a-number", "15"),
], ["encounter_id", "event_timestamp", "bill_amount", "wait_minutes"])

test_quality = (
    test_quality
    .withColumn("event_timestamp", F.to_timestamp("event_timestamp"))
    .withColumn("bill_amount", F.expr("try_cast(bill_amount AS DOUBLE)"))
    .withColumn("wait_minutes", F.col("wait_minutes").cast("int"))
)
assert test_quality.filter(
    (F.col("encounter_id") == "E-VALID") & F.col("event_timestamp").isNotNull()
    & (F.col("bill_amount") > 0) & (F.col("wait_minutes") >= 0)
).count() == 1, "Valid record test failed"
assert test_quality.filter(F.col("encounter_id") == "").count() == 1, "Invalid identifier test failed"
assert test_quality.filter(F.col("bill_amount").isNull()).count() == 1, "Invalid numeric parse test failed"

# Test B: duplicate business keys retain one deterministic winner.
test_duplicates = spark.createDataFrame([
    ("E-DUPLICATE", "2026-01-01 09:00:00", "hash-a"),
    ("E-DUPLICATE", "2026-01-01 10:00:00", "hash-b"),
], ["encounter_id", "ingest_timestamp", "_source_row_hash"])
test_duplicates = test_duplicates.withColumn("ingest_timestamp", F.to_timestamp("ingest_timestamp"))
dedup_test_window = Window.partitionBy("encounter_id").orderBy(
    F.col("ingest_timestamp").desc_nulls_last(), F.col("_source_row_hash").asc_nulls_last()
)
dedup_test_result = test_duplicates.withColumn(
    "_rn", F.row_number().over(dedup_test_window)
).filter(F.col("_rn") == 1)
assert dedup_test_result.count() == 1, "Duplicate deduplication test failed"
assert dedup_test_result.select("_source_row_hash").first()[0] == "hash-b", "Dedup tie-breaker test failed"

# Test C: deterministic latest sequence/time ordering.
test_events = spark.createDataFrame([
    ("E-DUP", 1, "OPEN", "2026-01-01 09:00:00"),
    ("E-DUP", 2, "COMPLETED", "2026-01-01 10:00:00"),
], ["encounter_id", "source_sequence", "new_status", "status_changed_at"])
test_events = test_events.withColumn("status_changed_at", F.to_timestamp("status_changed_at"))
test_window = Window.partitionBy("encounter_id").orderBy(
    F.col("source_sequence").desc(), F.col("status_changed_at").desc()
)
test_latest = test_events.withColumn("_rn", F.row_number().over(test_window)).filter(F.col("_rn") == 1)
assert test_latest.select("new_status").first()[0] == "COMPLETED", "Latest event ordering test failed"

# Test D: late arrival does not replace a newer source-sequence status.
# Input order is intentionally late/out-of-order.
test_late = spark.createDataFrame([
    ("E-LATE", 10, "COMPLETED", "2026-01-01 10:00:00"),
    ("E-LATE", 8, "OPEN", "2026-01-01 08:00:00"),
    ("E-LATE", 9, "PENDING", "2026-01-01 09:00:00"),
], ["encounter_id", "source_sequence", "new_status", "status_changed_at"])
test_late = test_late.withColumn("status_changed_at", F.to_timestamp("status_changed_at"))
late_window = Window.partitionBy("encounter_id").orderBy(
    F.col("source_sequence").desc(), F.col("status_changed_at").desc()
)
late_latest = test_late.withColumn("_rn", F.row_number().over(late_window)).filter(F.col("_rn") == 1)
assert late_latest.select("new_status").first()[0] == "COMPLETED", "Late event overwrote newer status"

# Test E: conflicting events with same sequence are detected.
test_conflict = spark.createDataFrame([
    ("E-CONFLICT", 5, "OPEN"),
    ("E-CONFLICT", 5, "CANCELLED"),
], ["encounter_id", "source_sequence", "new_status"])
conflict_count = (
    test_conflict.groupBy("encounter_id", "source_sequence")
    .agg(F.countDistinct("new_status").alias("versions"))
    .filter(F.col("versions") > 1)
    .count()
)
assert conflict_count == 1, "Conflicting sequence detection test failed"

print("PASS: valid record")
print("PASS: invalid mandatory/numeric fields")
print("PASS: duplicate deduplication and deterministic tie-breaker")
print("PASS: latest-event ordering")
print("PASS: late and out-of-order events")
print("PASS: conflicting source_sequence detection")
print("NOTE: Admission/transfer/discharge interval tests require confirmed event-type source fields.")

PASS: valid record
PASS: invalid mandatory/numeric fields
PASS: duplicate deduplication and deterministic tie-breaker
PASS: latest-event ordering
PASS: late and out-of-order events
PASS: conflicting source_sequence detection
NOTE: Admission/transfer/discharge interval tests require confirmed event-type source fields.


## 13. Final row-count and quality summary

In [0]:
# Compact final summary; detailed validation outputs are shown beside each processing section.
validation_rows = [
    ("Bronze encounter events", spark.table(BRONZE_ENCOUNTER_TABLE).count()),
    ("Silver encounter events", spark.table(SILVER_ENCOUNTER_TABLE).count()),
    ("Silver rejected records (including duplicate extras)", spark.table(SILVER_REJECTED_TABLE).count()),
    ("Bronze status CDC", spark.table(BRONZE_CDC_TABLE).count()),
    ("Silver status history", spark.table(SILVER_STATUS_HISTORY_TABLE).count()),
    ("Silver current encounters", spark.table(SILVER_CURRENT_TABLE).count()),
    ("Bronze secondary events", spark.table(BRONZE_SECONDARY_TABLE).count()),
    ("Silver secondary events", spark.table(SILVER_SECONDARY_TABLE).count()),
]
display(spark.createDataFrame(validation_rows, ["validation_check", "row_count"]))

print("Duplicate audit — input:", input_count,
      "| mandatory rejects:", rejected_mandatory_count,
      "| duplicate extras:", duplicate_count,
      "| Silver retained:", silver_count)

validation_check,row_count
Bronze encounter events,11540
Silver encounter events,11500
Silver rejected records (including duplicate extras),40
Bronze status CDC,520
Silver status history,520
Silver current encounters,11500
Bronze secondary events,310
Silver secondary events,310


Duplicate audit — input: 11540 | mandatory rejects: 0 | duplicate extras: 40 | Silver retained: 11500
